In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. COMMON FUNCTIONS
# ============================================================

def awgn(signal, snr_db):
    """Add AWGN noise to a signal."""
    signal_power = np.mean(signal ** 2)
    snr_linear = 10 ** (snr_db / 10)

    noise_power = signal_power / snr_linear
    noise = np.sqrt(noise_power) * np.random.randn(len(signal))

    return signal + noise


def bits_to_symbols(bits):
    """Convert bits to +/-1 symbols."""
    return 2 * bits - 1


# ============================================================
# 2. ASK TRANSMITTER
# ============================================================

def ask_transmitter(bits, fc, fs, Tb):
    """
    Binary ASK:
        bit 1 -> carrier
        bit 0 -> no carrier
    """

    samples_per_bit = int(fs * Tb)
    t_bit = np.arange(samples_per_bit) / fs

    carrier = np.cos(2 * np.pi * fc * t_bit)

    signal = []

    for bit in bits:
        if bit == 1:
            signal.extend(carrier)
        else:
            signal.extend(np.zeros(samples_per_bit))

    return np.array(signal)


# ============================================================
# 3. ASK COHERENT CORRELATOR RECEIVER
# ============================================================

def ask_coherent_receiver(received, fc, fs, Tb):
    """
    Coherent ASK detection using correlation with
    a synchronized carrier.
    """

    samples_per_bit = int(fs * Tb)
    t_bit = np.arange(samples_per_bit) / fs

    carrier = np.cos(2 * np.pi * fc * t_bit)

    detected_bits = []

    # Correlation threshold
    reference_energy = np.sum(carrier ** 2)
    threshold = reference_energy / 2

    for i in range(0, len(received), samples_per_bit):

        segment = received[i:i + samples_per_bit]

        correlation = np.sum(segment * carrier)

        if correlation > threshold:
            detected_bits.append(1)
        else:
            detected_bits.append(0)

    return np.array(detected_bits)


# ============================================================
# 4. BFSK TRANSMITTER
# ============================================================

def bfsk_transmitter(bits, f1, f2, fs, Tb):
    """
    Binary FSK:
        bit 0 -> f1
        bit 1 -> f2
    """

    samples_per_bit = int(fs * Tb)
    t_bit = np.arange(samples_per_bit) / fs

    signal = []

    for bit in bits:

        if bit == 0:
            tone = np.cos(2 * np.pi * f1 * t_bit)
        else:
            tone = np.cos(2 * np.pi * f2 * t_bit)

        signal.extend(tone)

    return np.array(signal)


# ============================================================
# 5. BFSK COHERENT CORRELATOR RECEIVER
# ============================================================

def bfsk_coherent_receiver(received, f1, f2, fs, Tb):
    """
    Coherent BFSK receiver.

    Correlate the received signal with both f1 and f2.
    The larger correlation determines the transmitted bit.
    """

    samples_per_bit = int(fs * Tb)
    t_bit = np.arange(samples_per_bit) / fs

    ref1 = np.cos(2 * np.pi * f1 * t_bit)
    ref2 = np.cos(2 * np.pi * f2 * t_bit)

    detected_bits = []

    for i in range(0, len(received), samples_per_bit):

        segment = received[i:i + samples_per_bit]

        corr1 = np.sum(segment * ref1)
        corr2 = np.sum(segment * ref2)

        if corr2 > corr1:
            detected_bits.append(1)
        else:
            detected_bits.append(0)

    return np.array(detected_bits)


# ============================================================
# 6. BFSK NONCOHERENT RECEIVER
# ============================================================

def bfsk_noncoherent_receiver(received, f1, f2, fs, Tb):
    """
    Noncoherent BFSK detection.

    Instead of requiring carrier phase synchronization,
    calculate energy at each frequency.

    Larger energy -> detected frequency.
    """

    samples_per_bit = int(fs * Tb)
    t_bit = np.arange(samples_per_bit) / fs

    ref1_cos = np.cos(2 * np.pi * f1 * t_bit)
    ref1_sin = np.sin(2 * np.pi * f1 * t_bit)

    ref2_cos = np.cos(2 * np.pi * f2 * t_bit)
    ref2_sin = np.sin(2 * np.pi * f2 * t_bit)

    detected_bits = []

    for i in range(0, len(received), samples_per_bit):

        segment = received[i:i + samples_per_bit]

        # Energy at f1
        I1 = np.sum(segment * ref1_cos)
        Q1 = np.sum(segment * ref1_sin)

        energy1 = I1 ** 2 + Q1 ** 2

        # Energy at f2
        I2 = np.sum(segment * ref2_cos)
        Q2 = np.sum(segment * ref2_sin)

        energy2 = I2 ** 2 + Q2 ** 2

        if energy2 > energy1:
            detected_bits.append(1)
        else:
            detected_bits.append(0)

    return np.array(detected_bits)


# ============================================================
# 7. BER FUNCTION
# ============================================================

def calculate_ber(original, detected):

    errors = np.sum(original != detected)

    return errors / len(original)


# ============================================================
# 8. MAIN PROGRAM
# ============================================================

np.random.seed(10)

# Number of transmitted bits
N = 1000

# Random binary data
bits = np.random.randint(0, 2, N)

# Sampling frequency
fs = 10000

# Bit duration
Tb = 0.01

# Carrier frequencies
fc = 500

# BFSK frequencies
f1 = 400
f2 = 500

# SNR
snr_db = 8


# ============================================================
# ASK TRANSMISSION
# ============================================================

ask_tx = ask_transmitter(
    bits,
    fc,
    fs,
    Tb
)

ask_rx_signal = awgn(
    ask_tx,
    snr_db
)

ask_detected = ask_coherent_receiver(
    ask_rx_signal,
    fc,
    fs,
    Tb
)

ask_ber = calculate_ber(
    bits,
    ask_detected
)


# ============================================================
# BFSK TRANSMISSION
# ============================================================

bfsk_tx = bfsk_transmitter(
    bits,
    f1,
    f2,
    fs,
    Tb
)

bfsk_rx_signal = awgn(
    bfsk_tx,
    snr_db
)


# ------------------------------------------------------------
# Coherent BFSK
# ------------------------------------------------------------

bfsk_coherent_detected = bfsk_coherent_receiver(
    bfsk_rx_signal,
    f1,
    f2,
    fs,
    Tb
)

bfsk_coherent_ber = calculate_ber(
    bits,
    bfsk_coherent_detected
)


# ------------------------------------------------------------
# Noncoherent BFSK
# ------------------------------------------------------------

bfsk_noncoherent_detected = bfsk_noncoherent_receiver(
    bfsk_rx_signal,
    f1,
    f2,
    fs,
    Tb
)

bfsk_noncoherent_ber = calculate_ber(
    bits,
    bfsk_noncoherent_detected
)


# ============================================================
# RESULTS
# ============================================================

print("======================================")
print("DIGITAL COMMUNICATION RESULTS")
print("======================================")

print("Number of bits :", N)
print("SNR            :", snr_db, "dB")

print("\nASK")
print("BER =", ask_ber)

print("\nBFSK Coherent")
print("BER =", bfsk_coherent_ber)

print("\nBFSK Noncoherent")
print("BER =", bfsk_noncoherent_ber)


# ============================================================
# 9. PLOT TRANSMITTED SIGNALS
# ============================================================

time_ask = np.arange(len(ask_tx)) / fs
time_bfsk = np.arange(len(bfsk_tx)) / fs

# Show only first few bits
show_bits = 5
show_samples = int(show_bits * fs * Tb)

plt.figure(figsize=(12, 5))

plt.plot(
    time_ask[:show_samples],
    ask_tx[:show_samples]
)

plt.title("ASK Transmitted Signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid()

plt.show()


plt.figure(figsize=(12, 5))

plt.plot(
    time_bfsk[:show_samples],
    bfsk_tx[:show_samples]
)

plt.title("BFSK Transmitted Signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid()

plt.show()